# Objective

Goal: identify high-confidence non-Australian indicators from sampled text in the AU subset, using rule-based methods such as regex and contextual heuristics.

Principle: prioritize precision over recall; avoid weak signals such as spelling differences or mere mentions of foreign topics.


## Load one parquet file as a sample DataFrame

This cell loads one parquet file from the AU subset into a pandas DataFrame called `df`.

We only load one part first because the goal at this stage is to test rules on a small sample, not to process the full dataset.

This creates the DataFrame required for later steps such as:

- checking column names
- applying regex rules
- inspecting matched examples

In [1]:
import cudf

print("cuDF version:", cudf.__version__)

cuDF version: 26.04.000


## Load one parquet file 

The previous read error was caused by using the wrong filesystem path.

This cell uses the corrected path under the current working directory to load one parquet file into a cuDF DataFrame.

In [8]:
file_path = "data/AUTokens50/part_0.parquet"
gdf = cudf.read_parquet(file_path)

print("Loaded file:", file_path)
print("Shape:", gdf.shape)
gdf.head()

Loaded file: data/AUTokens50/part_0.parquet
Shape: (278106, 9)


,text,id,dump,url,date,file_path,language,language_score,token_count
664,"Justine Davies –, Monday, January, 31, 2011, (...",<urn:uuid:0f3c1f06-4af1-4da0-96f9-91a7473f205f>,CC-MAIN-2013-20,http://blogs.news.com.au/moneystuff/index.php/...,2013-05-18T06:19:58Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.961377,1484
5457,Interview: Jenny Macpherson\n- by Rowena Scott...,<urn:uuid:12bb0bca-ddf8-4522-97ee-5583b8c93b93>,CC-MAIN-2013-20,http://www.bicycles.net.au/2010/10/interview-j...,2013-05-18T06:53:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.967261,2409
5459,The foundations for successful riding\n19 post...,<urn:uuid:74766fcf-d5eb-44fd-b925-01bd98098d70>,CC-MAIN-2013-20,http://www.bicycles.net.au/forums/viewtopic.ph...,2013-05-18T06:29:13Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.975258,2439
8507,photos by Carlo Ledesma\nWhen God was creating...,<urn:uuid:161ec30a-6da3-4d83-be97-32d4d9684b0c>,CC-MAIN-2013-20,http://www.kluster.com.au/issuefive/bees/,2013-05-18T07:25:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.961999,1419
9656,Joanne Harris is apparently as formidable a Yo...,<urn:uuid:2058b3ba-a9ad-4d81-80ab-4ce6b79c5d9e>,CC-MAIN-2013-20,http://www.northerndailyleader.com.au/story/97...,2013-05-18T08:10:31Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.990345,1806


## Create a small test subset without random sampling

The previous attempt used `sample()`, which depends on GPU random-number libraries.

Since the current environment is missing the required `curand` library, we avoid random sampling for now and simply take the first 5000 rows as a small working subset.

This is sufficient for early rule testing and manual inspection.

In [11]:
sample_size = min(5000, len(gdf))
sample_gdf = gdf.head(sample_size).copy()

print("Sample shape:", sample_gdf.shape)
sample_gdf.head()

Sample shape: (5000, 9)


,text,id,dump,url,date,file_path,language,language_score,token_count
664,"Justine Davies –, Monday, January, 31, 2011, (...",<urn:uuid:0f3c1f06-4af1-4da0-96f9-91a7473f205f>,CC-MAIN-2013-20,http://blogs.news.com.au/moneystuff/index.php/...,2013-05-18T06:19:58Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.961377,1484
5457,Interview: Jenny Macpherson\n- by Rowena Scott...,<urn:uuid:12bb0bca-ddf8-4522-97ee-5583b8c93b93>,CC-MAIN-2013-20,http://www.bicycles.net.au/2010/10/interview-j...,2013-05-18T06:53:36Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.967261,2409
5459,The foundations for successful riding\n19 post...,<urn:uuid:74766fcf-d5eb-44fd-b925-01bd98098d70>,CC-MAIN-2013-20,http://www.bicycles.net.au/forums/viewtopic.ph...,2013-05-18T06:29:13Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.975258,2439
8507,photos by Carlo Ledesma\nWhen God was creating...,<urn:uuid:161ec30a-6da3-4d83-be97-32d4d9684b0c>,CC-MAIN-2013-20,http://www.kluster.com.au/issuefive/bees/,2013-05-18T07:25:14Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.961999,1419
9656,Joanne Harris is apparently as formidable a Yo...,<urn:uuid:2058b3ba-a9ad-4d81-80ab-4ce6b79c5d9e>,CC-MAIN-2013-20,http://www.northerndailyleader.com.au/story/97...,2013-05-18T08:10:31Z,s3://commoncrawl/crawl-data/CC-MAIN-2013-20/se...,en,0.990345,1806


## Level1：Very strong signals

## Rule 1: US state abbreviation + ZIP code

This rule captures a strong US-local address pattern.

Examples include:

- `CA 90210`
- `NY 10001`
- `TX 73301-1234`

This is treated as a very strong non-Australian signal because it combines:

1. a valid US state abbreviation, and  
2. a valid US ZIP code format.

This pattern is unlikely to appear in genuine Australian local addresses.

At this stage, we use it as a high-confidence flag rather than an automatic deletion rule.

In [14]:
import re

US_STATES = r"(AL|AK|AZ|AR|CA|CO|CT|DE|FL|GA|HI|IA|ID|IL|IN|KS|KY|LA|MA|MD|ME|MI|MN|MO|MS|MT|NC|ND|NE|NH|NJ|NM|NV|NY|OH|OK|OR|PA|RI|SC|SD|TN|TX|UT|VA|VT|WA|WI|WV|WY)"

us_zip_state_pattern = re.compile(
    rf"\b{US_STATES}\s+\d{{5}}(?:-\d{{4}})?\b"
)

## Quick sanity check for the regex

This cell tests the regex on a few manually written examples before applying it to the sampled dataset.

The goal is to confirm that:

- US-style address patterns are matched correctly
- Australian-style address patterns are not matched
- general foreign references without address structure are not matched

In [15]:
test_texts = [
    "Our office is located at Los Angeles, CA 90210.",
    "Please mail the form to Albany, NY 10001.",
    "The Sydney office is in NSW 2000.",
    "This article discusses tourism in the United States.",
    "Ship returns to Austin, TX 73301-0215."
]

for text in test_texts:
    matched = bool(us_zip_state_pattern.search(text))
    print(matched, "|", text)

True | Our office is located at Los Angeles, CA 90210.
True | Please mail the form to Albany, NY 10001.
False | The Sydney office is in NSW 2000.
False | This article discusses tourism in the United States.
True | Ship returns to Austin, TX 73301-0215.


## Wrap the rule into a reusable function

This cell converts the regex into a reusable Python function.

The function returns:

- `True` if the text contains a US state + ZIP code pattern
- `False` otherwise

This makes it easier to apply the rule to a pandas DataFrame later.

In [16]:
def match_us_zip_state(text):
    if not isinstance(text, str):
        return False
    return bool(us_zip_state_pattern.search(text))

## Apply Rule 1 to the sampled text column

This cell applies Rule 1 to the `text` column of the sampled dataset.

A new boolean column is created:

- `rule_us_zip_state`

This column indicates whether each row contains a strong US-style address signal.

In [18]:
rule_us_zip_state_values = [
    match_us_zip_state(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_us_zip_state"] = rule_us_zip_state_values

## Count how many rows matched Rule 1

This cell counts how many rows in the sampled dataset were flagged by Rule 1.

This gives a quick overview of how frequent this strong US-style address signal is in the current sample.

In [20]:
sample_gdf["rule_us_zip_state"].value_counts()

rule_us_zip_state
False    4995
True        5
Name: count, dtype: int64

## Display the matched texts in full for manual inspection

The previous table shows which rows were matched, but the text content is truncated.

This cell prints the matched texts one by one so that we can inspect whether the rule is capturing:

- genuine US-style address patterns
- quoted foreign contact details
- or possible false positives inside Australian pages

In [ ]:
matched_rows = matched_us_zip_state.to_pandas()

for i, row in matched_rows.iterrows():
    print("=" * 100)
    print("Index:", i)
    print("URL:", row["url"])
    print("Date:", row["date"])
    print("Language score:", row["language_score"])
    print("Token count:", row["token_count"])
    print("Text:")
    print(row["text"])
    print()

In [ ]:
## Observation for Rule 1

the result above is not in this notebook because the result is too long.

Rule 1 matched only a very small number of rows in the 5000-row sample, which suggests that it is a strict and high-precision rule.

Manual inspection shows that the rule is correctly identifying genuine US-style address fragments such as:

- `Phoenix, AZ 85023-3409`
- `New York, NY 10019`
- `Ada, MI 49301`
- `Wasilla AK 99654-9720`
- `Chanhassen, MN 55317`

However, several matched rows still come from `.au` or `.com.au` pages. This indicates that the rule is effective at detecting strong foreign-local address signals, but not necessarily non-Australian webpages as a whole.

Therefore, Rule 1 should be treated as a **high-confidence non-AU signal flag**, rather than an automatic document-level deletion rule.

## Summary of Rule 1 evaluation

This rule was evaluated on a 5000-row sample.

We summarize both its strengths and its limitations below.

In [24]:
rule_1_summary = {
    "rule_name": "US state abbreviation + ZIP code",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_us_zip_state"].sum()),
    "strength": "Very high precision for US-style address fragments",
    "limitation": "Matched rows may still belong to Australian pages that mention foreign addresses",
    "recommended_use": "High-confidence flag, not automatic deletion"
}

rule_1_summary

{'rule_name': 'US state abbreviation + ZIP code',
 'sample_size': 5000,
 'matches': 5,
 'strength': 'Very high precision for US-style address fragments',
 'limitation': 'Matched rows may still belong to Australian pages that mention foreign addresses',
 'recommended_use': 'High-confidence flag, not automatic deletion'}

## Rule 2: Canadian postal code

This rule captures a strong Canadian local address pattern.

Examples include:

- `K1A 0B1`
- `M5V 3L9`
- `H2X 1Y4`

This is treated as a very strong non-Australian signal because Canadian postal codes have a highly distinctive alternating letter-number format.

This pattern is unlikely to appear naturally in genuine Australian local addresses.

At this stage, we use it as a high-confidence flag rather than an automatic deletion rule.

In [25]:
canada_postcode_pattern = re.compile(
    r"\b[ABCEGHJ-NPRSTVXY]\d[ABCEGHJ-NPRSTV-Z][ -]?\d[ABCEGHJ-NPRSTV-Z]\d\b",
    re.IGNORECASE
)

## Quick sanity check for Rule 2

Before applying the rule to the sampled dataset, we first test it on a few manually written examples.

The goal is to confirm that:

- Canadian postal codes are matched
- Australian postcodes are not matched
- general country mentions without postal-code structure are not matched

In [27]:
test_texts = [
    "Please send the parcel to Ottawa, ON K1A 0B1.",
    "The Toronto office is at M5V 3L9.",
    "The Sydney office is in NSW 2000.",
    "This article mentions Canada but gives no address.",
    "Mail can be sent to Montreal, QC H2X 1Y4."
]

for text in test_texts:
    matched = bool(canada_postcode_pattern.search(text))
    print(matched, "|", text)

True | Please send the parcel to Ottawa, ON K1A 0B1.
True | The Toronto office is at M5V 3L9.
False | The Sydney office is in NSW 2000.
False | This article mentions Canada but gives no address.
True | Mail can be sent to Montreal, QC H2X 1Y4.


## Wrap Rule 2 into a reusable function

This cell wraps the Canadian postal-code regex into a reusable Python function.

The function returns:

- `True` if a text contains a Canadian postal code
- `False` otherwise

This makes it easier to apply the rule to the sampled dataset later.

In [28]:
def match_canada_postcode(text):
    if not isinstance(text, str):
        return False
    return bool(canada_postcode_pattern.search(text))

## Apply Rule 2 to the sampled text column

This cell applies Rule 2 to the `text` column of the sampled dataset.

A new boolean column is created:

- `rule_canada_postcode`

This column indicates whether each row contains a strong Canadian postal-code signal.

In [29]:
rule_canada_postcode_values = [
    match_canada_postcode(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_canada_postcode"] = rule_canada_postcode_values

## Count how many rows matched Rule 2

This cell counts how many rows in the sampled dataset were flagged by Rule 2.

This provides a quick overview of how frequent the Canadian postal-code signal is in the current sample.

In [30]:
sample_gdf["rule_canada_postcode"].value_counts()

rule_canada_postcode
False    5000
Name: count, dtype: int64

## Observation for Rule 2

Rule 2 matched zero rows in the current 5000-row sample.

This suggests that Canadian postal-code patterns are either absent or extremely rare in this subset.

As a result, no further manual inspection was necessary for this rule at this stage.

## Summary of Rule 2 evaluation

This rule was evaluated on the current 5000-row sample.

We summarize the result below.

In [31]:
rule_2_summary = {
    "rule_name": "Canadian postal code",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_canada_postcode"].sum()),
    "strength": "Very high precision for Canadian postal-code fragments",
    "limitation": "No matched examples observed in the current sample",
    "recommended_use": "Keep as a high-confidence flag for larger-scale testing"
}

rule_2_summary

{'rule_name': 'Canadian postal code',
 'sample_size': 5000,
 'matches': 0,
 'strength': 'Very high precision for Canadian postal-code fragments',
 'limitation': 'No matched examples observed in the current sample',
 'recommended_use': 'Keep as a high-confidence flag for larger-scale testing'}

## Rule 3: UK postcode

This rule captures a strong UK local address pattern.

Examples include:

- `SW1A 1AA`
- `EC1A 1BB`
- `M1 1AE`

This is treated as a very strong non-Australian signal because UK postcodes follow a distinctive alphanumeric structure that is very different from Australian postcodes.

At this stage, we use it as a high-confidence flag rather than an automatic deletion rule.

In [32]:
uk_postcode_pattern = re.compile(
    r"\b[A-Z]{1,2}\d[A-Z\d]?\s\d[A-Z]{2}\b",
    re.IGNORECASE
)

## Quick sanity check for Rule 3

Before applying the rule to the sampled dataset, we first test it on a few manually written examples.

The goal is to confirm that:

- UK postcodes are matched
- Australian postcodes are not matched
- general UK mentions without postcode structure are not matched

In [33]:
test_texts = [
    "The London office is at SW1A 1AA.",
    "Please send the document to EC1A 1BB.",
    "The Manchester site uses M1 1AE.",
    "The Sydney office is in NSW 2000.",
    "This article mentions the UK but gives no address."
]

for text in test_texts:
    matched = bool(uk_postcode_pattern.search(text))
    print(matched, "|", text)

True | The London office is at SW1A 1AA.
True | Please send the document to EC1A 1BB.
True | The Manchester site uses M1 1AE.
False | The Sydney office is in NSW 2000.
False | This article mentions the UK but gives no address.


## Wrap Rule 3 into a reusable function

This cell wraps the UK postcode regex into a reusable Python function.

In [37]:
def match_uk_postcode(text):
    if not isinstance(text, str):
        return False
    return bool(uk_postcode_pattern.search(text))

## Apply Rule 3 to the sampled text column

This cell applies Rule 3 to the `text` column of the sampled dataset.

A new boolean column is created:

- `rule_uk_postcode`

In [38]:
rule_uk_postcode_values = [
    match_uk_postcode(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_uk_postcode"] = rule_uk_postcode_values

In [39]:
sample_gdf["rule_uk_postcode"].value_counts()

rule_uk_postcode
False    4997
True        3
Name: count, dtype: int64

## Observation for Rule 1,2, 3

We can clearly see this time there is still very little data which is has Uk postcode. We combine rule1 to rule3, for the parquet0, we can almost say that the 50 bilion tokens are almost all australian data. Which means the workload for manual check is acceptable. By checking the postcode we have some "maybe non-aus data", and we can manual check it. However, maybe later we can still do some code to check these "maybe non-aus data" after checking the whole data by using the postcode rule.

## Summary of Rule 3 evaluation

This cell records the evaluation result of Rule 3 in a structured form for later comparison with other rules.

In [61]:
rule_3_summary = {
    "rule_name": "UK postcode",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_uk_postcode"].sum()),
    "strength": "Very high precision for UK postcode fragments",
    "limitation": "A matched postcode does not necessarily mean the whole page is non-Australian",
    "recommended_use": "High-confidence flag, not automatic deletion"
}

rule_3_summary

{'rule_name': 'UK postcode',
 'sample_size': 5000,
 'matches': 3,
 'strength': 'Very high precision for UK postcode fragments',
 'limitation': 'A matched postcode does not necessarily mean the whole page is non-Australian',
 'recommended_use': 'High-confidence flag, not automatic deletion'}

## Rule 4: Foreign institution terms

This rule captures strongly localized foreign institutional terms.

Examples include:

- `IRS`
- `Social Security Number`
- `SSN`
- `DMV`
- `NHS`
- `HMRC`

These terms are treated as strong non-Australian signals because they refer to country-specific institutions or systems that do not naturally belong to Australian local administrative contexts.

At this stage, we use them as high-confidence flags rather than automatic deletion rules.

In [40]:
foreign_institution_pattern = re.compile(
    r"\b(IRS|Social Security Number|SSN|DMV|NHS|HMRC|National Insurance)\b",
    re.IGNORECASE
)

## Quick sanity check for Rule 4

Before applying the rule to the sampled dataset, we first test it on a few manually written examples.

The goal is to confirm that:

- foreign institutional terms are matched
- ordinary Australian text without such terms is not matched

In [42]:
test_texts = [
    "Please contact the IRS about your tax records.",
    "You may need your Social Security Number for this form.",
    "The NHS provides health services in the UK.",
    "HMRC has updated its guidance.",
    "This is an Australian article about local schools."
]

for text in test_texts:
    matched = bool(foreign_institution_pattern.search(text))
    print(matched, "|", text)

True | Please contact the IRS about your tax records.
True | You may need your Social Security Number for this form.
True | The NHS provides health services in the UK.
True | HMRC has updated its guidance.
False | This is an Australian article about local schools.


## Wrap Rule 4 into a reusable function

This cell wraps the foreign-institution regex into a reusable Python function.

In [44]:
def match_foreign_institution(text):
    if not isinstance(text, str):
        return False
    return bool(foreign_institution_pattern.search(text))

## Apply Rule 4 to the sampled text column

This cell applies Rule 4 to the `text` column of the sampled dataset.

A new boolean column is created:

- `rule_foreign_institution`

In [46]:
rule_foreign_institution_values = [
    match_foreign_institution(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_foreign_institution"] = rule_foreign_institution_values

## Count how many rows matched Rule 4

This cell counts how many rows in the sampled dataset were flagged by Rule 4.

In [48]:
sample_gdf["rule_foreign_institution"].value_counts()

rule_foreign_institution
False    4980
True       20
Name: count, dtype: int64

## Inspect the rows matched by Rule 4

This cell displays the rows flagged by Rule 4 for manual inspection.

The goal is to verify whether the matched rows really contain strong foreign institutional terms, and to identify whether they appear in:

- foreign forms or service pages
- Australian pages discussing foreign systems
- quoted or copied foreign administrative content

In [49]:
matched_foreign_institution = sample_gdf[sample_gdf["rule_foreign_institution"]][
    ["text", "url", "date", "language_score", "token_count"]
]

matched_foreign_institution

,text,url,date,language_score,token_count
763370,"Wednesday, March 31, 2010\nI have recently sof...",http://caseymulligan.blogspot.com.au/2010_03_0...,2013-05-19T02:59:00Z,0.971302,12018
67887,You should also think about the leg extension ...,http://www.injuryupdate.com.au/forum/showthrea...,2013-05-21T17:25:07Z,0.980411,3515
242046,Identity thieves go shopping for Apple product...,http://www.theage.com.au/it-pro/security-it/id...,2013-05-20T02:08:29Z,0.964759,1192
522283,Mother refuses to let son undergo radiotherapy...,http://www.news.com.au/lifestyle/health-fitnes...,2013-06-19T14:18:30Z,0.983004,1280
949515,Australian Bureau of Statistics\n1301.0 - Year...,http://www.abs.gov.au/Ausstats/abs@.nsf/Previo...,2013-05-25T20:17:32Z,0.965995,1773
13658,"The embedded video, from Saraqeb in the Idlib ...",http://israelmatzav.blogspot.com.au/,2013-05-21T10:20:56Z,0.961052,11466
334706,Midwives face legal uncertainty for homebirths...,http://www.abc.net.au/radionational/programs/l...,2013-05-23T11:56:24Z,0.976422,6448
684819,View Full Version : Special needs chat!\ni don...,http://www.bubhub.com.au/community/forums/arch...,2013-05-21T10:49:57Z,0.972190,26645
822178,Confessions of a hypochondriac\n“Just because ...,http://www.thepunch.com.au/articles/confession...,2013-05-19T19:21:38Z,0.973403,1745
1077965,anyone still out there in conservative land?\n...,http://www.injuryupdate.com.au/forum/showthrea...,2013-05-24T16:38:43Z,0.977747,1396


## Display the matched texts in full for Rule 4

The table view truncates the text column, so this cell prints the matched rows one by one.

This makes it easier to inspect whether the rule is capturing:

- genuine foreign institutional terms
- Australian pages discussing foreign systems
- copied or quoted foreign administrative language

In [ ]:
matched_rows = matched_foreign_institution.to_pandas()

for i, row in matched_rows.iterrows():
    print("=" * 100)
    print("Index:", i)
    print("URL:", row["url"])
    print("Date:", row["date"])
    print("Language score:", row["language_score"])
    print("Token count:", row["token_count"])
    print("Text:")
    print(row["text"])
    print()

## False-positive analysis for Rule 4

Again, the result is too long. I didn't put the result on. But, the result is not good.

Manual inspection shows that Rule 4 produces a noticeable number of false positives.

The main reasons are:

1. **Some matched terms are too short and too generic**
   Acronyms such as `IRS` and `NHS` can be matched in contexts that are not true foreign administrative references, or may appear in quoted text without indicating page origin.

2. **Australian pages may still mention foreign institutions**
   Several matched rows come from `.au` or `.com.au` pages, but the text includes discussion of foreign systems such as US tax or UK health administration. In these cases, the rule correctly detects a foreign reference, but this does **not** imply that the whole page is non-Australian.

3. **Quoted or imported foreign content can trigger matches**
   News articles, commentary, blogs, and forum discussions may reproduce foreign administrative language, even when the hosting page is Australian.

Overall, Rule 4 is useful for detecting **strong foreign-local references**, but it is too broad to be used as an automatic document-level exclusion rule in its current form.

A more precise version should focus on longer and more explicit phrases, such as:

- `Social Security Number`
- `Internal Revenue Service`
- `Department of Motor Vehicles`
- `National Insurance`
- `HM Revenue and Customs`

This would reduce false positives and improve precision.

## Refine Rule 4 to reduce false positives

Manual inspection shows that the original institution-term rule is too broad.

In particular, very short acronyms such as `IRS` and `NHS` may create false positives when matched without enough context.

This refined version focuses on more explicit institutional phrases in order to improve precision.

In [51]:
foreign_institution_pattern_v2 = re.compile(
    r"\b("
    r"Social Security Number|"
    r"Internal Revenue Service|"
    r"Department of Motor Vehicles|"
    r"National Insurance|"
    r"HM Revenue and Customs"
    r")\b",
    re.IGNORECASE
)

## Wrap the refined Rule 4 into a reusable function

This cell wraps the refined institution-term regex into a reusable Python function.

In [53]:
def match_foreign_institution_v2(text):
    if not isinstance(text, str):
        return False
    return bool(foreign_institution_pattern_v2.search(text))

## Apply the refined Rule 4 to the sampled text column

This cell applies the refined institution-term rule to the sampled dataset.

A new boolean column is created:

- `rule_foreign_institution_v2`

In [54]:
rule_foreign_institution_v2_values = [
    match_foreign_institution_v2(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_foreign_institution_v2"] = rule_foreign_institution_v2_values

## Count how many rows matched the refined Rule 4

This cell counts how many rows in the sampled dataset were flagged by the refined institution-term rule.

The goal is to check whether the stricter version reduces false positives compared with the original Rule 4.

In [56]:
sample_gdf["rule_foreign_institution_v2"].value_counts()

rule_foreign_institution_v2
False    4994
True        6
Name: count, dtype: int64

## Observation for refined Rule 4

The refined version of Rule 4 matched 6 rows in the current 5000-row sample, compared with 20 matches from the original version.

This suggests that replacing short acronyms with longer and more explicit institutional phrases substantially reduces false positives.

The refined rule therefore appears to have higher precision than the original institution-term rule.

However, as with the other Level 1 rules, a match should still be treated as a strong foreign-local signal rather than automatic evidence that the whole page is non-Australian.

## Summary of refined Rule 4 evaluation

This cell records the evaluation result of the refined institution-term rule in a structured form.

In [57]:
rule_4_v2_summary = {
    "rule_name": "Foreign institution terms (refined)",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_foreign_institution_v2"].sum()),
    "previous_matches": int(sample_gdf["rule_foreign_institution"].sum()),
    "strength": "Higher precision than the original institution-term rule",
    "limitation": "A matched term may still appear inside Australian pages that discuss foreign systems",
    "recommended_use": "High-confidence flag, not automatic deletion"
}

rule_4_v2_summary

{'rule_name': 'Foreign institution terms (refined)',
 'sample_size': 5000,
 'matches': 6,
 'previous_matches': 20,
 'strength': 'Higher precision than the original institution-term rule',
 'limitation': 'A matched term may still appear inside Australian pages that discuss foreign systems',
 'recommended_use': 'High-confidence flag, not automatic deletion'}

## Select the rows matched by the refined Rule 4

This cell extracts only the rows flagged by the refined institution-term rule.

The resulting subset will be used for manual inspection.

In [59]:
matched_foreign_institution_v2 = sample_gdf[sample_gdf["rule_foreign_institution_v2"]][
    ["text", "url", "date", "language_score", "token_count"]
]

matched_foreign_institution_v2

,text,url,date,language_score,token_count
242046,Identity thieves go shopping for Apple product...,http://www.theage.com.au/it-pro/security-it/id...,2013-05-20T02:08:29Z,0.964759,1192
13658,"The embedded video, from Saraqeb in the Idlib ...",http://israelmatzav.blogspot.com.au/,2013-05-21T10:20:56Z,0.961052,11466
812873,"Exclusive Brethren heavyweights\nfrom left, Ph...",http://www.smh.com.au/news/national/the-influe...,2013-06-19T06:49:10Z,0.975805,3948
67229,Women in Parliament\nThe idea that women would...,http://www.parliament.vic.gov.au/about/the-his...,2013-05-21T17:43:21Z,0.984234,2456
146078,Skip to content\n22 May 2013\nThe end of a fir...,http://insurancenews.com.au/analysis/victorias...,2013-05-22T07:34:07Z,0.961347,1598
73445,Who would want to be a millionaire? Not even m...,http://www.camdencourier.com.au/story/928998/a...,2013-05-24T02:05:20Z,0.960364,2975


## Display the matched texts in full for the refined Rule 4

The table view truncates the text column, so this cell prints the matched rows one by one.

This makes it easier to inspect whether the refined rule is capturing explicit institutional phrases rather than broad acronyms.

In [ ]:
matched_rows_v2 = matched_foreign_institution_v2.to_pandas()

for i, row in matched_rows_v2.iterrows():
    print("=" * 100)
    print("Index:", i)
    print("URL:", row["url"])
    print("Date:", row["date"])
    print("Language score:", row["language_score"])
    print("Token count:", row["token_count"])
    print("Text:")
    print(row["text"])
    print()

## Observation for refined Rule 4

I didn't put the result because it's too long

Manual inspection suggests that the refined Rule 4 is substantially cleaner than the original version.

The refined rule captures explicit foreign institutional references such as:

- `Social Security number`
- `IRS`
- `NHS`
- `National Insurance`

However, the matched rows show that these references often appear inside Australian pages, including `.com.au`, `.gov.au`, and `.net.au` sites, especially in news reporting, policy discussion, or comparative commentary.

Therefore, the refined rule has better precision than the original version, but it still should not be used as an automatic document-level exclusion rule.

Instead, it is best treated as a **high-confidence foreign-local signal flag**.

## Level 1 summary table

This table summarizes the evaluation results of all Level 1 high-confidence rules tested on the current 5000-row sample.

The goal is to compare:

- how often each rule matched
- how precise each rule appears to be
- whether each rule is suitable for automatic filtering or only for flagging

In [65]:
level_1_summary = [
    rule_1_summary,
    rule_2_summary,
    rule_3_summary,
    rule_4_v2_summary
]

level_1_summary_gdf = cudf.DataFrame(level_1_summary)
level_1_summary_gdf

,rule_name,sample_size,matches,strength,limitation,recommended_use,previous_matches
0,US state abbreviation + ZIP code,5000,5,Very high precision for US-style address fragm...,Matched rows may still belong to Australian pa...,"High-confidence flag, not automatic deletion",<NA>
1,Canadian postal code,5000,0,Very high precision for Canadian postal-code f...,No matched examples observed in the current sa...,Keep as a high-confidence flag for larger-scal...,<NA>
2,UK postcode,5000,3,Very high precision for UK postcode fragments,A matched postcode does not necessarily mean t...,"High-confidence flag, not automatic deletion",<NA>
3,Foreign institution terms (refined),5000,6,Higher precision than the original institution...,A matched term may still appear inside Austral...,"High-confidence flag, not automatic deletion",20.0


## Display the Level 1 summary as a table

This cell converts the Level 1 summary into a tabular format for easier comparison across rules.

In [66]:
level_1_summary_gdf[
    ["rule_name", "sample_size", "matches", "recommended_use"]
]

,rule_name,sample_size,matches,recommended_use
0,US state abbreviation + ZIP code,5000,5,"High-confidence flag, not automatic deletion"
1,Canadian postal code,5000,0,Keep as a high-confidence flag for larger-scal...
2,UK postcode,5000,3,"High-confidence flag, not automatic deletion"
3,Foreign institution terms (refined),5000,6,"High-confidence flag, not automatic deletion"


## Level 1 overall conclusion

The Level 1 rules are useful for detecting **very strong foreign-local signals** in the sampled AU subset.

Across the current 5000-row sample, postcode-based rules and explicit institution-term rules matched only a small number of rows, which suggests that they are relatively strict and high-precision patterns.

However, manual inspection shows an important limitation: a matched foreign postcode or institution term does **not** necessarily mean that the whole page is non-Australian. In many cases, Australian pages legitimately mention foreign addresses, institutions, legal systems, or public services in reporting, commentary, or imported content.

Therefore, the Level 1 rules should be treated as:

- **high-confidence signal flags**
- useful for **candidate detection**
- suitable for **manual review or downstream scoring**

but **not** as automatic document-level deletion rules.

Among the tested rules, the refined institution-term rule performed better than the original broader version, indicating that longer and more explicit phrases improve precision and reduce false positives.

##













## Level 2: Strong but contextual signals

Level 2 rules are designed to capture signals that are still useful for identifying potentially non-Australian content, but are less definitive than Level 1 rules.

Unlike Level 1, these patterns are not usually sufficient on their own. They need more context, because they may appear in:

- international news reporting
- comparisons between countries
- imported or syndicated content
- Australian pages discussing foreign topics

Therefore, Level 2 rules should be treated as:

- contextual signals
- medium-confidence flags
- useful for ranking, review, or combined scoring

rather than standalone evidence for document-level exclusion.

Typical Level 2 candidates include:

- US phone number formats
- explicit mentions of `ZIP code`
- combinations such as city + US state name
- foreign service phrases such as `Social Security Administration`
- contact or office-page language tied to foreign locations

## Difference between Level 1 and Level 2

Level 1 rules focus on highly distinctive foreign-local patterns, such as foreign postcode formats or explicit institutional phrases.

Level 2 rules focus on patterns that are still informative, but require additional context to interpret correctly.

In practice:

- Level 1 is mainly for high-confidence flagging
- Level 2 is mainly for contextual evidence and downstream scoring

This distinction helps avoid over-filtering while still preserving useful signals for identifying potentially non-Australian content.

## Level 2 Rule 1: Explicit mention of `ZIP code`

This rule captures explicit mentions of the term `ZIP code`.

Unlike Level 1 postcode rules, this rule does not identify a full address pattern. Instead, it detects a US-specific postal-system term.

This makes it a **contextual foreign-local signal** rather than a definitive exclusion signal.

Examples include:

- `Enter your ZIP code`
- `Please provide a valid ZIP code`
- `Search by ZIP code`

This rule should be treated as a medium-confidence flag rather than an automatic deletion rule.

In [67]:
zip_code_term_pattern = re.compile(
    r"\bZIP code\b",
    re.IGNORECASE
)

## Wrap Level 2 Rule 1 into a reusable function

This cell wraps the `ZIP code` regex into a reusable Python function.

In [69]:
def match_zip_code_term(text):
    if not isinstance(text, str):
        return False
    return bool(zip_code_term_pattern.search(text))

## Apply Level 2 Rule 1 to the sampled text column

This cell applies the `ZIP code` term rule to the sampled dataset.

A new boolean column is created:

- `rule_zip_code_term`

In [70]:
rule_zip_code_term_values = [
    match_zip_code_term(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_zip_code_term"] = rule_zip_code_term_values

## Count how many rows matched Level 2 Rule 1

This cell counts how many rows in the sampled dataset were flagged by the `ZIP code` term rule.

In [72]:
sample_gdf["rule_zip_code_term"].value_counts()

rule_zip_code_term
False    4999
True        1
Name: count, dtype: int64

## Inspect the row matched by Level 2 Rule 1

This cell displays the row flagged by the `ZIP code` term rule for manual inspection.

The goal is to verify whether the match is a meaningful US-specific contextual signal.

In [73]:
matched_zip_code_term = sample_gdf[sample_gdf["rule_zip_code_term"]][
    ["text", "url", "date", "language_score", "token_count"]
]

matched_zip_code_term

,text,url,date,language_score,token_count
109024,I don’t think I’ve gone a few weeks without ha...,http://www.therefinedgeek.com.au/index.php/tag...,2013-05-22T00:29:51Z,0.970718,1177


## Observation for Level 2 Rule 1

Level 2 Rule 1 matched 1 row in the current 5000-row sample.

This suggests that explicit mentions of `ZIP code` are rare in the sampled AU subset, and therefore may serve as a useful low-frequency contextual signal.

However, the matched example comes from a `.com.au` page, which shows that Australian-hosted pages may still refer to US-specific postal terminology.

Therefore, this rule should be treated as a **medium-confidence contextual flag** rather than as evidence for automatic document-level exclusion.

## Summary of Level 2 Rule 1 evaluation

This cell records the evaluation result of the `ZIP code` term rule in a structured form for later comparison with other Level 2 rules.

In [74]:
level_2_rule_1_summary = {
    "rule_name": "Explicit mention of ZIP code",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_zip_code_term"].sum()),
    "strength": "Useful low-frequency US-specific contextual signal",
    "limitation": "May appear in Australian-hosted pages that discuss US systems",
    "recommended_use": "Medium-confidence flag, not automatic deletion"
}

level_2_rule_1_summary

{'rule_name': 'Explicit mention of ZIP code',
 'sample_size': 5000,
 'matches': 1,
 'strength': 'Useful low-frequency US-specific contextual signal',
 'limitation': 'May appear in Australian-hosted pages that discuss US systems',
 'recommended_use': 'Medium-confidence flag, not automatic deletion'}

## Level 2 Rule 2: US phone number formats

This rule captures common US phone number formats.

Examples include:

- `(212) 555-1234`
- `212-555-1234`
- `212 555 1234`
- `+1 212 555 1234`

Unlike Level 1 rules, this rule does not by itself prove that a page is non-Australian. Australian pages may mention foreign contact numbers, imported content, or international office details.

Therefore, this rule should be treated as a medium-confidence contextual flag rather than an automatic deletion rule.

In [75]:
us_phone_pattern = re.compile(
    r"\b(?:\+1[\s-]?)?(?:\(?\d{3}\)?[\s-]?)\d{3}[\s-]?\d{4}\b"
)

## Wrap Level 2 Rule 2 into a reusable function

This cell wraps the US phone-number regex into a reusable Python function.

In [76]:
def match_us_phone(text):
    if not isinstance(text, str):
        return False
    return bool(us_phone_pattern.search(text))

## Apply Level 2 Rule 2 to the sampled text column

This cell applies the US phone-number rule to the sampled dataset.

A new boolean column is created:

- `rule_us_phone`

In [77]:
rule_us_phone_values = [
    match_us_phone(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_us_phone"] = rule_us_phone_values

## Count how many rows matched Level 2 Rule 2

This cell counts how many rows in the sampled dataset were flagged by the US phone-number rule.

In [78]:
sample_gdf["rule_us_phone"].value_counts()

rule_us_phone
False    4982
True       18
Name: count, dtype: int64

## Inspect the rows matched by Level 2 Rule 2

This cell displays the rows flagged by the US phone-number rule for manual inspection.

The goal is to check whether the matched phone-number patterns are genuine US-style contact numbers or false positives caused by generic numeric sequences.

In [80]:
matched_us_phone = sample_gdf[sample_gdf["rule_us_phone"]][
    ["text", "url", "date", "language_score", "token_count"]
]

matched_us_phone

,text,url,date,language_score,token_count
149896,"Workshop tales, trials and disasters.\nMainten...",http://www.bicycles.net.au/forums/viewtopic.ph...,2013-05-19T10:04:48Z,0.968202,1823
1038518,"Intellegent excellent working parents, Father ...",http://www.gumtree.com.au/s-other-pets/c18436?...,2013-06-18T05:26:19Z,0.971932,1242
297254,"We are Dane and Fraser, we share a funky 2 bed...",http://www.gumtree.com.au/s-couchsurfing/page-...,2013-05-23T04:27:33Z,0.964592,2031
640445,|Home | Bookmark | Tell||Active petitions in o...,http://www.gopetition.com.au/tag/television/page5,2013-05-20T22:24:24Z,0.967956,4512
795636,Banks of the old bayou\nSouthern-style ... the...,http://www.theage.com.au/travel/banks-of-the-o...,2013-05-22T08:20:51Z,0.961463,2131
208636,|Home | Bookmark | Tell||Active petitions in o...,http://www.gopetition.com.au/tag/american,2013-05-24T22:43:33Z,0.960674,5618
404358,Clary Fray just wishes that her life would go ...,http://www.burnbright.com.au/burn-bright-book-...,2013-05-18T17:27:56Z,0.965496,1799
829537,Her Majesty's pleasure\nSaska Graville does ti...,http://www.canberratimes.com.au/travel/her-maj...,2013-05-24T23:21:04Z,0.964494,1120
839605,|Home | Bookmark | Tell||Active petitions in o...,http://www.gopetition.com.au/tag/minnesota,2013-06-19T13:13:23Z,0.972912,5047
641842,The home of 'cool'\n23 posts • Page 1 of 1\nG ...,http://www.bicycles.net.au/forums/viewtopic.ph...,2013-05-23T11:34:13Z,0.960696,1481


## Display the matched texts in full for Level 2 Rule 2

The table view truncates the text column, so this cell prints the matched rows one by one.

This makes it easier to inspect whether the rule is capturing genuine US phone numbers or false positives caused by generic numeric patterns.

In [ ]:
matched_us_phone_rows = matched_us_phone.to_pandas()

for i, row in matched_us_phone_rows.iterrows():
    print("=" * 100)
    print("Index:", i)
    print("URL:", row["url"])
    print("Date:", row["date"])
    print("Language score:", row["language_score"])
    print("Token count:", row["token_count"])
    print("Text:")
    print(row["text"])
    print()

## False-positive analysis for Level 2 Rule 2

Manual inspection shows that the original US phone-number rule produces many false positives.

The main reasons are:

1. **Australian phone numbers are being matched**
   Many matched rows contain Australian mobile or local contact numbers rather than US phone numbers.

2. **Non-US international phone numbers are also being matched**
   Some rows contain UK or Australian international numbers, which indicates that the pattern is too broad and is not sufficiently distinguishing US-specific formats.

3. **Australian pages may include genuine foreign contact numbers**
   Even when the matched number is a genuine US-style number, it may appear inside an Australian-hosted page such as a news article, travel page, forum, or petition page.

Overall, the original rule is too broad to be used as a reliable US-phone detector. A refined version should place more emphasis on explicitly US-linked formats such as:

- `+1` country code
- US-style parentheses format, e.g. `(212) 555-1234`
- stronger context such as nearby US location terms

Therefore, the current version should not be used as a filtering rule in its present form.

## Refine Level 2 Rule 2 to reduce false positives

Manual inspection shows that the original US phone-number rule is too broad.

It matches many non-US numeric patterns, including:

- Australian mobile numbers
- Australian local phone numbers
- non-US international phone numbers

The refined version focuses on more explicitly US-linked formats, especially:

- numbers with the `+1` country code
- numbers using the typical US parentheses style, such as `(212) 555-1234`

This refined rule is expected to improve precision.

In [82]:
us_phone_pattern_v2 = re.compile(
    r"(?:\+1[\s-]\d{3}[\s-]\d{3}[\s-]\d{4})|(?:\(\d{3}\)\s?\d{3}[-]\d{4})"
)

## Wrap the refined Level 2 Rule 2 into a reusable function

This cell wraps the refined US phone-number regex into a reusable Python function.

In [83]:
def match_us_phone_v2(text):
    if not isinstance(text, str):
        return False
    return bool(us_phone_pattern_v2.search(text))

## Apply the refined Level 2 Rule 2 to the sampled text column

This cell applies the refined US phone-number rule to the sampled dataset.

A new boolean column is created:

- `rule_us_phone_v2`

In [84]:
rule_us_phone_v2_values = [
    match_us_phone_v2(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_us_phone_v2"] = rule_us_phone_v2_values

## Count how many rows matched the refined Level 2 Rule 2

This cell counts how many rows in the sampled dataset were flagged by the refined US phone-number rule.

The goal is to compare the refined version with the original broad version.

In [85]:
sample_gdf["rule_us_phone_v2"].value_counts()

rule_us_phone_v2
False    4997
True        3
Name: count, dtype: int64

## Observation for refined Level 2 Rule 2

The refined US phone-number rule matched 3 rows in the current 5000-row sample, compared with 18 matches from the original version.

This suggests that restricting the pattern to more explicitly US-linked formats substantially reduces false positives.

The refined rule therefore appears to have much higher precision than the original broad phone-number rule.

However, as with other Level 2 rules, a matched US-style phone number may still appear inside Australian-hosted pages such as travel pages, petitions, or news-related content.

Therefore, the refined rule should be treated as a **medium-confidence contextual flag** rather than an automatic document-level exclusion rule.

## Summary of refined Level 2 Rule 2 evaluation

This cell records the evaluation result of the refined US phone-number rule in a structured form for later comparison with other Level 2 rules.

In [86]:
level_2_rule_2_v2_summary = {
    "rule_name": "US phone number formats (refined)",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_us_phone_v2"].sum()),
    "previous_matches": int(sample_gdf["rule_us_phone"].sum()),
    "strength": "Higher precision than the original broad phone-number rule",
    "limitation": "May still appear in Australian-hosted pages that include foreign contact details",
    "recommended_use": "Medium-confidence flag, not automatic deletion"
}

level_2_rule_2_v2_summary

{'rule_name': 'US phone number formats (refined)',
 'sample_size': 5000,
 'matches': 3,
 'previous_matches': 18,
 'strength': 'Higher precision than the original broad phone-number rule',
 'limitation': 'May still appear in Australian-hosted pages that include foreign contact details',
 'recommended_use': 'Medium-confidence flag, not automatic deletion'}

## Inspect the rows matched by the refined Level 2 Rule 2

This cell displays the rows flagged by the refined US phone-number rule for manual inspection.

The goal is to verify whether the refined rule is now capturing more explicit and more reliable US-linked phone-number formats.

In [87]:
matched_us_phone_v2 = sample_gdf[sample_gdf["rule_us_phone_v2"]][
    ["text", "url", "date", "language_score", "token_count"]
]

matched_us_phone_v2

,text,url,date,language_score,token_count
640445,|Home | Bookmark | Tell||Active petitions in o...,http://www.gopetition.com.au/tag/television/page5,2013-05-20T22:24:24Z,0.967956,4512
795636,Banks of the old bayou\nSouthern-style ... the...,http://www.theage.com.au/travel/banks-of-the-o...,2013-05-22T08:20:51Z,0.961463,2131
839605,|Home | Bookmark | Tell||Active petitions in o...,http://www.gopetition.com.au/tag/minnesota,2013-06-19T13:13:23Z,0.972912,5047


## Manual inspection for refined Level 2 Rule 2

Manual inspection shows that the refined US phone-number rule is substantially cleaner than the original version.

The remaining matched rows appear to be cases where the text contains more explicit US-linked contact details, such as petition pages or travel-related pages that include overseas phone numbers.

However, these rows can still come from Australian-hosted pages. This means that even the refined rule should be interpreted as a contextual foreign-local signal rather than as automatic evidence that the whole page is non-Australian.

Overall, the refined version has much better precision than the original phone-number rule and is more suitable for downstream scoring or review.

## Important interpretation note

A page hosted on an Australian domain is not necessarily Australian in content context.

For example, an Australian-hosted page may primarily describe:

- US institutions
- US contact details
- US services
- US administrative conventions

Such pages may still be part of the Australian web ecosystem, so they should not automatically be discarded.

However, they may be less useful if the goal is to build a dataset focused on **Australian local content and context**, rather than simply **Australian-hosted pages**.

Therefore, the purpose of the rules in this notebook is not only to detect non-Australian pages, but also to identify pages that are **Australian-hosted but strongly foreign in content context**.

## Level 2 Rule 3: Unambiguous US-style date format

This rule captures date strings that are unambiguously in US format:

- `MM/DD/YYYY`

but only when the day value is greater than 12, so that the pattern cannot be confused with the Australian `DD/MM/YYYY` format.

Examples include:

- `05/21/2013`
- `12/31/2024`

This rule is treated as a **medium-confidence contextual signal** rather than an automatic deletion rule, because Australian-hosted pages may still quote or reproduce US-formatted dates.

In [89]:
us_date_pattern = re.compile(
    r"\b(?:0?[1-9]|1[0-2])/(?:1[3-9]|2[0-9]|3[01])/(?:19|20)\d{2}\b"
)

## Wrap Level 2 Rule 3 into a reusable function

This cell wraps the unambiguous US-style date regex into a reusable Python function.

In [90]:
def match_us_date(text):
    if not isinstance(text, str):
        return False
    return bool(us_date_pattern.search(text))

## Apply Level 2 Rule 3 to the sampled text column

This cell applies the unambiguous US-style date rule to the sampled dataset.

A new boolean column is created:

- `rule_us_date`

In [91]:
rule_us_date_values = [
    match_us_date(text) for text in sample_gdf["text"].to_pandas()
]

sample_gdf["rule_us_date"] = rule_us_date_values

## Count how many rows matched Level 2 Rule 3

This cell counts how many rows in the sampled dataset were flagged by the unambiguous US-style date rule.

In [92]:
sample_gdf["rule_us_date"].value_counts()

rule_us_date
False    4998
True        2
Name: count, dtype: int64

## Inspect the rows matched by Level 2 Rule 3

This cell displays the rows flagged by the unambiguous US-style date rule for manual inspection.

The goal is to verify whether the matched dates are meaningful US-style contextual signals rather than accidental numeric patterns.

In [93]:
matched_us_date = sample_gdf[sample_gdf["rule_us_date"]][
    ["text", "url", "date", "language_score", "token_count"]
]

matched_us_date

,text,url,date,language_score,token_count
454569,|Home | Bookmark | Tell||Active petitions in o...,http://www.gopetition.com.au/tag/relationships,2013-05-22T00:43:25Z,0.961787,2083
108049,Louisiana Ammo Bunker Explosion 10/17/2012\nRu...,http://americankabuki.blogspot.com.au/2012/10/...,2013-05-24T08:51:23Z,0.978185,1248


In [94]:
matched_us_date_rows = matched_us_date.to_pandas()

for i, row in matched_us_date_rows.iterrows():
    print("=" * 100)
    print("Index:", i)
    print("URL:", row["url"])
    print("Date:", row["date"])
    print("Language score:", row["language_score"])
    print("Token count:", row["token_count"])
    print("Text:")
    print(row["text"])
    print()

Index: 454569
URL: http://www.gopetition.com.au/tag/relationships
Date: 2013-05-22T00:43:25Z
Language score: 0.9617870450019836
Token count: 2083
Text:
|Home | Bookmark | Tell||Active petitions in over 75 countries||Follow GoPetition|
Petition Tag - relationships
Facebook does not allow Users to indicate they are in a Committed Relationship. They do, however, allow Users to indicate they are in a Casual Relationship or in an Open Relationship.
Facebook removed the ability to designate a Family Member as a "Partner". They used to allow this, and should allow it, as not all relationships are as casual as "in a relationship" implies.
It is wrong of Facebook to allow the designation of "In an Open Relationship" but not to recognize committed, though non-legal, partnerships. They even go so far as to distinguish "Cousin (male)" from "Cousin (female)", yet they can't include an option for Partner.
Check out this website http://www.ojp.usdoj.gov/nij/journals/256/sexual-assault.html
Women are 

## Observation for Level 2 Rule 3

Level 2 Rule 3 matched a small number of rows in the current 5000-row sample.

This suggests that unambiguous US-style date formats are relatively rare, but still useful as contextual foreign-local signals.

The matched examples show that such dates can appear inside Australian-hosted pages, which means that the rule should not be interpreted as direct evidence that the whole page is non-Australian.

Therefore, this rule is best treated as a **medium-confidence contextual flag** rather than an automatic document-level exclusion rule.

## Summary of Level 2 Rule 3 evaluation

This cell records the evaluation result of the unambiguous US-style date rule in a structured form for later comparison with other Level 2 rules.

In [95]:
level_2_rule_3_summary = {
    "rule_name": "Unambiguous US-style date format",
    "sample_size": 5000,
    "matches": int(sample_gdf["rule_us_date"].sum()),
    "strength": "Useful low-frequency US-style contextual signal",
    "limitation": "May appear inside Australian-hosted pages that quote or reproduce foreign content",
    "recommended_use": "Medium-confidence flag, not automatic deletion"
}

level_2_rule_3_summary

{'rule_name': 'Unambiguous US-style date format',
 'sample_size': 5000,
 'matches': 2,
 'strength': 'Useful low-frequency US-style contextual signal',
 'limitation': 'May appear inside Australian-hosted pages that quote or reproduce foreign content',
 'recommended_use': 'Medium-confidence flag, not automatic deletion'}

## Level 2 summary table

This section summarizes the evaluation results of the Level 2 rules tested on the current 5000-row sample.

The summary is displayed as a cuDF DataFrame so that the results are easier to compare in table form.

In [96]:
level_2_summary = [
    level_2_rule_1_summary,
    level_2_rule_2_v2_summary,
    level_2_rule_3_summary
]

level_2_summary_gdf = cudf.DataFrame(level_2_summary)
level_2_summary_gdf

,rule_name,sample_size,matches,strength,limitation,recommended_use,previous_matches
0,Explicit mention of ZIP code,5000,1,Useful low-frequency US-specific contextual si...,May appear in Australian-hosted pages that dis...,"Medium-confidence flag, not automatic deletion",<NA>
1,US phone number formats (refined),5000,3,Much higher precision than the original broad ...,Matched numbers may still appear inside Austra...,"Medium-confidence flag, not automatic deletion",18.0
2,Unambiguous US-style date format,5000,2,Useful low-frequency US-style contextual signal,May appear inside Australian-hosted pages that...,"Medium-confidence flag, not automatic deletion",<NA>


## Level 2 overall conclusion

The Level 2 rules are useful for detecting **contextual foreign-local signals** that are weaker than Level 1 rules but still informative.

Across the current 5000-row sample, these rules matched only a small number of rows, which suggests that they can help identify potentially non-Australian content context without being overwhelmingly broad.

However, manual inspection consistently shows that these signals often appear inside Australian-hosted pages, including news pages, petition pages, blogs, travel pages, and other local sites that quote or reproduce foreign content.

Therefore, Level 2 rules should not be treated as automatic document-level filtering rules.

Instead, they are best used as:

- **medium-confidence contextual flags**
- signals for **manual review**
- features for **downstream scoring or ranking**

This helps distinguish between:

- pages that are Australian-hosted
- pages that are strongly Australian in content context
- and pages that are Australian-hosted but contain strong foreign-local references